In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df

partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# select needed columns and compute TOTAL_COST on GPU
partsupp_filtered = partsupp[["PS_PARTKEY","PS_SUPPKEY","PS_SUPPLYCOST","PS_AVAILQTY"]]
partsupp_filtered["TOTAL_COST"] = \
    partsupp_filtered["PS_SUPPLYCOST"] * partsupp_filtered["PS_AVAILQTY"]

# select supplier keys
supplier_filtered = supplier[["S_SUPPKEY","S_NATIONKEY"]]

# join partsupp with supplier and keep only needed columns
ps_supp_merge = (
    partsupp_filtered
    .merge(
        supplier_filtered,
        left_on="PS_SUPPKEY",
        right_on="S_SUPPKEY",
        how="inner"
    )
    [["PS_PARTKEY","S_NATIONKEY","TOTAL_COST"]]
)

# filter nation to Germany and keep only key
nation_filtered = nation[nation["N_NAME"] == "GERMANY"][["N_NATIONKEY"]]

# join with filtered nation and keep only partkey and cost
ps_supp_n_merge = (
    ps_supp_merge
    .merge(
        nation_filtered,
        left_on="S_NATIONKEY",
        right_on="N_NATIONKEY",
        how="inner"
    )
    [["PS_PARTKEY","TOTAL_COST"]]
)

# compute threshold
sum_val = ps_supp_n_merge["TOTAL_COST"].sum() * 0.0001

# group on GPU, sum and rename without NamedAgg
total = (
    ps_supp_n_merge
    .groupby("PS_PARTKEY", sort=False)["TOTAL_COST"]
    .sum()
    .reset_index()
)
total.columns = ["PS_PARTKEY","VALUE"]

# filter and sort
total = total[total["VALUE"] > sum_val].sort_values("VALUE", ascending=False)